# E3 · Monotonic constraints ruin the model (or is it the order?)

**Type**: Find the bug · **Level**: ●●● challenge · **Time**: about 50 minutes · **extended path**

## Objective

A colleague adds `monotone_constraints=(-1, 0, 1, 0)` to a loan repayment model ("more income, more probability of repayment; more debt, less") and finds that the AUC collapses from 0.83 to 0.65: they conclude that "monotonic constraints do not work". Find the bug (the tuple follows a **different order** from the columns), fix it with a dictionary by name and
**check** that the model really is monotonic and that `income` has an effect again.

## Dataset

Synthetic data: 4000 applicants with `income`, `debt`, `age`, `noise`; repayment probability increasing with `income`, decreasing with `debt`, almost independent of `age`. Columns in the order `income, debt, age, noise`; 60/40 split.

## Educational purpose

Learn that `monotone_constraints` as a tuple is **positional** and does not check the names: a constraint in the wrong order is applied to another feature, forcing an effect opposite to the real one. And how to verify a constraint: a `profile` function that varies one feature keeping the others fixed.

## Task

1. Run the colleague's cell: `free_auc` (without constraints), `bug_auc` (with the colleague's tuple).
2. Diagnosis: `applied` (dictionary column → constraint actually applied, with `zip(X.columns, tuple)`); `real_signs` (dictionary column → sign of the correlation with `y`, `-1` or `+1`, for `income`, `debt`, `age`); `wrong` (sorted list of the columns whose applied constraint is `+1` or `-1` **and** has the opposite sign to the real one).
3. Fix: `ok_constraints = {"income": 1, "debt": -1}` (dictionary by name, accepted by XGBoost 1.6) and `ok_auc` with the constrained model.
4. Check with `profile(model, X_te, column)` (given: probability as one column varies, the others at the median): `bug_swing` and `ok_swing` (maximum minus minimum of the `income` profile in the two models: how much the probability changes as `income` varies), `free_violations` (number of grid points where, in the model **without** constraints, `income` makes the probability decrease) and `monotone_ok` (True if the corrected model has no violations for `income` (+) and `debt` (−)).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
n = 4000
X = pd.DataFrame({"income": rng.normal(size=n), "debt": rng.normal(size=n), "age": rng.normal(size=n), "noise": rng.normal(size=n)})
y = rng.binomial(1, 1 / (1 + np.exp(-(1.5 * X["income"] - 1.0 * X["debt"] + 0.3 * X["age"])))).astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)


def model(**kw):
    return xgb.XGBClassifier(n_estimators=150, learning_rate=0.1, max_depth=3, eval_metric="auc", n_jobs=-1, random_state=0, **kw)


def profile(m, X_ref, column, points=30):
    """Probability as one column varies (from the 5th to the 95th percentile), with the other columns at the median."""
    grid = np.linspace(X_ref[column].quantile(0.05), X_ref[column].quantile(0.95), points)
    base = pd.DataFrame([X_ref.median()] * points).reset_index(drop=True)
    base[column] = grid
    return m.predict_proba(base)[:, 1]

## Your work

### The colleague's code

In [ ]:
constraints = (-1, 0, 1, 0)            # debt (-), age, income (+), noise: 'I wrote them in the order I thought of the features'
free_model = model().fit(X_tr, y_tr)
bug_model = model(monotone_constraints=constraints).fit(X_tr, y_tr)
free_auc = roc_auc_score(y_te, free_model.predict_proba(X_te)[:, 1])
bug_auc = roc_auc_score(y_te, bug_model.predict_proba(X_te)[:, 1])
print(f"AUC without constraints {free_auc:.3f} | with the colleague's constraints {bug_auc:.3f} -> 'monotonic constraints ruin the model'")

### Your diagnosis and the fix

In [ ]:
applied = {}
real_signs = {}
wrong = []

In [ ]:
ok_constraints = None
ok_auc = None

In [ ]:
bug_swing = None
ok_swing = None
free_violations = None
monotone_ok = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`applied = dict(zip(X_tr.columns, constraints))`; signs: `np.sign(np.corrcoef(X_tr[c], y_tr)[0, 1])`. Swing: `p = profile(m, X_te, "income"); p.max() - p.min()`. Violations: `(np.diff(p) < -1e-9).sum()`. The corrected model: `model(monotone_constraints={"income": 1, "debt": -1}).fit(X_tr, y_tr)`.'):
    check(free_auc > 0.8 and bug_auc < free_auc - 0.1, "without constraints the AUC goes over 0.8; with the colleague's it loses more than 0.1", "don't modify the colleague's code")
    check(applied == {"income": -1, "debt": 0, "age": 1, "noise": 0}, "the tuple applied to the columns: income -1, debt 0, age +1, noise 0", "dict(zip(X.columns, tuple))")
    check(real_signs.get("income") == 1 and real_signs.get("debt") == -1 and real_signs.get("age") == 1, "real signs: income +, debt -, age +", "np.sign(np.corrcoef(X[c], y)[0, 1])")
    check(wrong == ["income"], "the only column with a constraint of the opposite sign to the real one is income (for age the sign matches by chance)", "applied constraint ≠ real sign, both different from 0")
    check(ok_constraints == {"income": 1, "debt": -1} and ok_auc is not None and ok_auc > free_auc - 0.01, "with the dictionary by name the AUC goes back to the free one (or exceeds it)", "monotone_constraints={'income': 1, 'debt': -1}")
    check(bug_swing is not None and bug_swing < 0.02 and ok_swing > 0.3, "in the colleague's model the probability is flat with respect to income (swing below 0.02); in the corrected one it changes by over 0.3", "profile(m, X_te, 'income').max() - .min()")
    check(free_violations is not None and free_violations > 0 and monotone_ok is True, "the free model has points where income makes the probability drop (spurious oscillations); the corrected one is monotonic for income (+) and debt (−)", "np.diff(profile(...))")
done()

---
The commented solution is in `solutions/E_find_the_bug/E3_monotone_constraints_in_the_wrong_order.ipynb`. Open it after you pass the check, or after you have really tried.